## 检查6张表格的内容和缺漏

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### 1. DA_codebook_full.xlsx

个人信息标签（包含心理测试表）：301个

生理指标检测标签：95个

### 2. DA_daily_diet.xlsx

这份数据主要记录了200名戒毒人员每日摄入的营养成分和热量总和。

- 数据缺失：DA163

- 并非所有戒毒人员都拥有完整的15天餐食记录。其中：

    8名人员仅记录了1-2天
    
    57名人员记录了8天

    134名人员拥有完整的15天数据



In [ ]:
dd_path = r"C:\Users\adayc\Desktop\code\Drug\DA_data\DA_daily_diet.xlsx"
dd = pd.read_excel(dd_path)

# get column names
column_names = dd.columns.tolist()
print(f"Got {len(column_names)-3} features. They are: {column_names[3:]}.")

# check ID column
existing_ids = set(dd[dd.columns[1]].unique())
expected_ids = set(range(1, 201)) 
missing_ids = sorted(expected_ids - existing_ids)
print(f"Missing {len(missing_ids)} DA_ID：{missing_ids}.")

Got 22 features. They are: ['s_能量千卡', 's_蛋白质克', 's_脂肪克', 's_碳水化物克', 's_膳食纤维克', 's_胆固醇毫克', 's_维生素A视黄醇当量μg', 's_维生素B1毫克', 's_维生素B2毫克', 's_烟酸毫克', 's_维生素C毫克', 's_维生素E毫克', 's_钙毫克', 's_磷毫克', 's_钾毫克', 's_钠毫克', 's_镁毫克', 's_铁毫克', 's_锌毫克', 's_硒ug', 's_铜毫克', 's_锰毫克'].
Missing 1 DA_ID：[163].


In [ ]:
# Record days count
participant_days = dd.groupby(dd.columns[0])[dd.columns[2]].nunique()
days_count = participant_days.value_counts().sort_index()

# Record days not 8 or 15 days
less_than_5_days = participant_days[(participant_days != 8) & (participant_days != 15)].reset_index()
less_than_5_days.columns = ["ID", "Record days"]
stats_df = pd.DataFrame({'Record days': days_count.index, 'Num. participants': days_count.values})

print("Table of record days：")
print(stats_df.to_string(index=False))
print("\nRecord IDs with days not 8 or 15 days：")
print(less_than_5_days.to_string(index=False))

Table of record days：
 Record days  Num. participants
           3                  1
           5                  1
           7                  2
           8                 57
          10                  1
          11                  1
          13                  1
          14                  1
          15                134

Record IDs with days not 8 or 15 days：
   ID  Record days
DA091            5
DA104            7
DA106            7
DA148           10
DA171           11
DA183           13
DA184           14
DA196            3


### 3. DA_daily_food_clean.xlsx

这份表格主要记录了200名戒毒人员每日的饮食情况，包括食物名称、进食时间、食用量，以及食物的能量和营养价值。

- 无数据缺失 -> 可以补充表1缺失的DA163

- 并非所有戒毒人员都拥有完整的15天餐食记录。其中：

    8名人员仅记录了1-2天
    
    57名人员记录了8天

    134名人员拥有完整的15天数据

In [27]:
dfc = pd.read_excel(r'C:\Users\adayc\Desktop\code\Drug\DA_data\DA_daily_food_clean.xlsx')
#print(dfc.info())

# Check missing ID
missing_data = dfc[dfc['Sampleid'].apply(lambda x: not f"DA{x[2:]}" in [f"DA{i:03d}" for i in range(1, 201)])]
print(f"Missing ID: {len(missing_data)}")

Missing ID: 0


In [ ]:
# Check record days
days_ount = dfc.groupby('Sampleid')['DAY'].nunique().reset_index(name='Days Count')
print("Record days for each participant:")
print(meal_counts)

# 3. Check meal time (4 meals/day)
complete_meal = dfc.groupby(['Sampleid', 'DAY'])['TIME'].nunique()
incomplete_meal = complete_meal[complete_meal < 4]
print("没有完整记录4个进食时间的记录:")
print(incomplete_meal)

Record days for each participant:
     Sampleid         DAY  Meal Count
0       DA001  10.29(星期二)           9
1       DA001  10.30(星期三)           9
2       DA001  10.31(星期四)          10
3       DA001   11.1(星期五)           7
4       DA001  11.10(星期天)           7
...       ...         ...         ...
2575    DA200  11.25(星期一)          10
2576    DA200  11.26(星期二)          10
2577    DA200  11.27(星期三)          10
2578    DA200  11.28(星期四)          10
2579    DA200  11.29(星期五)          10

[2580 rows x 3 columns]
没有完整记录4个进食时间的记录:
Sampleid  DAY       
DA001     11.11(星期一)    3
          11.12(星期二)    3
          11.5(星期二)     3
          11.9(星期六)     3
DA002     11.12(星期二)    3
                       ..
DA200     11.25(星期一)    3
          11.26(星期二)    3
          11.27(星期三)    3
          11.28(星期四)    3
          11.29(星期五)    3
Name: TIME, Length: 1038, dtype: int64


In [ ]:
# Record days count
participant_days_dfc = dfc.groupby('Sampleid')['DAY'].nunique()
days_count_dfc = participant_days_dfc.value_counts().sort_index()
stats_df_dfc = pd.DataFrame({'Record days': days_count_dfc.index, 'Num. participants': days_count_dfc.values})

print("Table of record days：")
print(stats_df_dfc.to_string(index=False))

# Record days not 8 or 15 days
less_than_5_days_dfc = participant_days_dfc[(participant_days_dfc != 8) & (participant_days_dfc != 15)].reset_index()
less_than_5_days_dfc.columns = ["ID", "Record days"]

print("\nRecord IDs with days not 8 or 15 days：")
print(less_than_5_days_dfc.to_string(index=False))

Table of record days：
 Record days  Num. participants
           8                 60
          15                140

Record IDs with days not 8 or 15 days：
Empty DataFrame
Columns: [ID, Record days]
Index: []


In [34]:
less_than_5_days

,index,Sampleid,Unique Days Count
0,0,DA001,NaN
1,1,DA002,NaN
2,2,DA003,NaN
3,3,DA004,NaN
4,4,DA005,NaN
...,...,...,...
195,195,DA196,NaN
196,196,DA197,NaN
197,197,DA198,NaN
198,198,DA199,NaN
